In [0]:
%python
# ============================================================================
# GDPR COMPLIANCE PIPELINE - BATCH TRANSLATION
# Complete pipeline: PDF ingestion → parsing → chunking → translation
# Processes 50 documents per run, tracks progress, fully resumable
# ============================================================================

from pyspark.sql.functions import col, expr, explode, concat_ws, struct, array_sort, collect_list, lit
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType
import re

# ============================================================================
# CONFIGURATION
# ============================================================================
VOLUME_PATH = "/Volumes/main/default/enforcement_tracker"
RAW_TABLE = "main.default.gdpr_raw_multilingual_precedents"
CHUNKS_TABLE = "main.default.gdpr_translated_chunks"
FINAL_TABLE = "main.default.gdpr_historical_fines"

BATCH_SIZE = 100  # Process 300 documents per run (~1 hour)
MAX_TOKENS = 1200  # Proven safe from testing (no truncation)

print("="*70)
print("GDPR TRANSLATION PIPELINE - BATCH PROCESSOR")
print("="*70)

# ============================================================================
# STEP 1: ENSURE RAW TABLE EXISTS (First-time setup)
# ============================================================================
print("\n[STEP 1] Checking raw document table...")

try:
    raw_exists = spark.table(RAW_TABLE).count()
    print(f"✓ Raw table exists: {raw_exists} documents")
except:
    print("⚠️ Raw table not found - creating from PDFs...")
    
    # Read PDFs from volume
    raw_binary_df = (spark.read
        .format("binaryFile")
        .option("pathGlobFilter", "*.pdf")
        .load(VOLUME_PATH))
    
    pdf_count = raw_binary_df.count()
    print(f"  Found {pdf_count} PDFs in volume")
    
    # Parse PDFs with AI
    print("  Parsing PDFs (this may take a while)...")
    parsed_df = raw_binary_df.withColumn("ai_output", expr("ai_parse_document(content)"))
    
    # Extract text payload
    final_df = parsed_df.select(
        expr("element_at(split(path, '/'), -1)").alias("source_file_name"),
        expr("""
            coalesce(
                cast(ai_output:document.metadata.page_count as int),
                array_max(
                    transform(
                        from_json(
                            cast(ai_output:document.elements as string),
                            'array<struct<page_number:int>>'
                        ),
                        x -> x.page_number
                    )
                ),
                1
            )
        """).alias("total_pages"),
        expr("""
            concat_ws('\\n\\n',
                transform(
                    from_json(
                        cast(ai_output:document.elements as string),
                        'array<struct<content:string>>'
                    ),
                    x -> x.content
                )
            )
        """).alias("raw_text_payload")
    )
    
    # Save to table
    (final_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(RAW_TABLE))
    
    print(f"✓ Raw table created: {pdf_count} documents")

# ============================================================================
# STEP 2: CHECK TRANSLATION PROGRESS
# ============================================================================
print("\n[STEP 2] Checking translation progress...")

try:
    processed_df = spark.table(CHUNKS_TABLE)
    already_processed = [row.source_file_name for row in processed_df.select("source_file_name").distinct().collect()]
    print(f"✓ Found {len(already_processed)} already-translated documents")
except:
    already_processed = []
    print("✓ No previous translations found - starting fresh")

# ============================================================================
# STEP 3: SELECT NEXT BATCH
# ============================================================================
print("\n[STEP 3] Loading documents for processing...")

all_docs_df = spark.table(RAW_TABLE)
total_docs = all_docs_df.count()

# Filter out already-processed
if already_processed:
    unprocessed_df = all_docs_df.filter(~col("source_file_name").isin(already_processed))
else:
    unprocessed_df = all_docs_df

remaining_docs = unprocessed_df.count()

print(f"  Total documents: {total_docs}")
print(f"  Already processed: {len(already_processed)}")
print(f"  Remaining: {remaining_docs}")

if remaining_docs == 0:
    print("\n🎉 ALL DOCUMENTS ALREADY PROCESSED!")
    print("Skipping to final reassembly...")
    skip_translation = True
else:
    skip_translation = False
    batch_df = unprocessed_df.limit(BATCH_SIZE)
    batch_count = batch_df.count()
    
    print(f"\n{'='*70}")
    print(f"PROCESSING BATCH: {batch_count} documents")
    print(f"Remaining after this batch: {remaining_docs - batch_count}")
    print(f"{'='*70}")

# ============================================================================
# STEP 4: CHUNKING FUNCTION
# ============================================================================
if not skip_translation:
    print("\n[STEP 4] Setting up chunking...")
    
    def estimate_tokens(text):
        if not text:
            return 0
        words = len(text.split())
        return int(words * 1.25)
    
    def paragraph_based_chunking(text):
        if not text:
            return []
        
        if estimate_tokens(text) <= MAX_TOKENS:
            return [{"chunk_index": 0, "chunk_text": text, "chunk_tokens": estimate_tokens(text)}]
        
        paragraphs = [p.strip() for p in re.split(r'\n\n+', text) if p.strip()]
        
        chunks = []
        current_chunk = []
        current_tokens = 0
        chunk_index = 0
        
        for para in paragraphs:
            para_tokens = estimate_tokens(para)
            
            if para_tokens > MAX_TOKENS:
                if current_chunk:
                    chunks.append({
                        "chunk_index": chunk_index,
                        "chunk_text": "\n\n".join(current_chunk),
                        "chunk_tokens": current_tokens
                    })
                    chunk_index += 1
                    current_chunk = []
                    current_tokens = 0
                
                lines = [line.strip() for line in para.split('\n') if line.strip()]
                for line in lines:
                    line_tokens = estimate_tokens(line)
                    if current_tokens + line_tokens <= MAX_TOKENS:
                        current_chunk.append(line)
                        current_tokens += line_tokens
                    else:
                        if current_chunk:
                            chunks.append({
                                "chunk_index": chunk_index,
                                "chunk_text": "\n".join(current_chunk),
                                "chunk_tokens": current_tokens
                            })
                            chunk_index += 1
                        current_chunk = [line]
                        current_tokens = line_tokens
                continue
            
            if current_tokens + para_tokens <= MAX_TOKENS:
                current_chunk.append(para)
                current_tokens += para_tokens
            else:
                if current_chunk:
                    chunks.append({
                        "chunk_index": chunk_index,
                        "chunk_text": "\n\n".join(current_chunk),
                        "chunk_tokens": current_tokens
                    })
                    chunk_index += 1
                current_chunk = [para]
                current_tokens = para_tokens
        
        if current_chunk:
            chunks.append({
                "chunk_index": chunk_index,
                "chunk_text": "\n\n".join(current_chunk),
                "chunk_tokens": current_tokens
            })
        
        return chunks
    
    # Register UDF
    chunk_schema = ArrayType(StructType([
        StructField("chunk_index", IntegerType(), False),
        StructField("chunk_text", StringType(), False),
        StructField("chunk_tokens", IntegerType(), False)
    ]))
    
    spark.udf.register("paragraph_chunk_udf", paragraph_based_chunking, chunk_schema)
    print("✓ Chunking function registered")

# ============================================================================
# STEP 5: CHUNK THE BATCH
# ============================================================================
if not skip_translation:
    print("\n[STEP 5] Chunking documents...")
    
    chunked_df = batch_df.withColumn(
        "chunks",
        expr("paragraph_chunk_udf(raw_text_payload)")
    ).withColumn(
        "chunk_exploded",
        explode(col("chunks"))
    ).select(
        col("source_file_name"),
        col("total_pages"),
        col("chunk_exploded.chunk_index").alias("chunk_index"),
        col("chunk_exploded.chunk_text").alias("chunk_text"),
        col("chunk_exploded.chunk_tokens").alias("chunk_tokens")
    )
    
    total_chunks = chunked_df.count()
    print(f"✓ Created {total_chunks} chunks from {batch_count} documents")
    print(f"⏱️  Estimated translation time: {total_chunks * 20 // 60} minutes")

# ============================================================================
# STEP 6: TRANSLATE THE BATCH
# ============================================================================
if not skip_translation:
    print(f"\n[STEP 6] {'='*70}")
    print("TRANSLATING CHUNKS (AI_TRANSLATE)")
    print(f"{'='*70}")
    
    translated_df = chunked_df.withColumn(
        "translated_text",
        expr("ai_translate(chunk_text, 'en')")
    )
    
    print("✓ Translation complete!")

# ============================================================================
# STEP 7: SAVE PROGRESS
# ============================================================================
if not skip_translation:
    print("\n[STEP 7] Saving progress...")
    
    if already_processed:
        # Append to existing table
        (translated_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(CHUNKS_TABLE))
    else:
        # Create new table
        (translated_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(CHUNKS_TABLE))
    
    print("✓ Batch saved to chunks table")

# ============================================================================
# STEP 8: PROGRESS SUMMARY
# ============================================================================
total_processed = spark.table(CHUNKS_TABLE).select("source_file_name").distinct().count()

print(f"\n{'='*70}")
print("BATCH SUMMARY")
print(f"{'='*70}")
if not skip_translation:
    print(f"Documents processed this batch: {batch_count}")
print(f"Total documents processed: {total_processed}/{total_docs}")
print(f"Remaining: {total_docs - total_processed}")
print(f"Progress: {total_processed/total_docs*100:.1f}%")

if total_processed < total_docs:
    print(f"\n👉 RE-RUN THIS CELL to process the next batch")
    print(f"   ({(total_docs - total_processed) // BATCH_SIZE + 1} batches remaining)")
else:
    print("\n✓ All documents translated! Assembling final table...")
    
    # ========================================================================
    # STEP 9: REASSEMBLE FINAL DOCUMENTS
    # ========================================================================
    print("\n[STEP 9] Reassembling translated documents...")
    
    final_docs = spark.table(CHUNKS_TABLE).groupBy("source_file_name").agg(
        expr("concat_ws('\\n\\n', transform(array_sort(collect_list(struct(chunk_index, translated_text))), x -> x.translated_text))").alias("translated_text_english")
    )
    
    # Join with original data
    raw_df = spark.table(RAW_TABLE)
    final_table = raw_df.join(final_docs, "source_file_name").select(
        "source_file_name",
        "total_pages",
        "raw_text_payload",
        "translated_text_english"
    )
    
    # Save final table
    (final_table.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FINAL_TABLE))
    
    print(f"✓ Final table created: {FINAL_TABLE}")
    print(f"✓ {total_processed} documents fully translated and assembled")
    print("\n🎉 PIPELINE COMPLETE!")

print("="*70)

In [0]:
# Check what got translated before hitting quota
translated_count = spark.table("main.default.gdpr_translated_chunks") \
    .select("source_file_name").distinct().count()

chunks_saved = spark.table("main.default.gdpr_translated_chunks").count()

print(f"✅ Documents translated: {translated_count}/6")
print(f"✅ Chunks saved: {chunks_saved}")
print("\nThese are saved - they won't be re-translated when you resume!")